# Day 39 – Q2: Aspect-Level Sentiment Analysis

Review-level F1=88% vs Aspect-level F1=71%. Why the gap, how to close it, and extracting aspect-sentiment pairs from a multi-aspect review.

In [3]:
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

REVIEWS_PATH = "shopsense_reviews.csv"
TARGET_REVIEW = "Amazing camera quality but the battery is atrocious and customer support was unhelpful."

ASPECT_KEYWORDS = {
    "camera":          ["camera", "photo", "picture", "image", "lens", "resolution"],
    "battery":         ["battery", "charge", "charging", "drain", "power", "backup"],
    "delivery":        ["delivery", "shipping", "dispatch", "courier", "arrived", "received"],
    "quality":         ["quality", "build", "material", "finish", "durable", "sturdy", "flimsy"],
    "price":           ["price", "cost", "value", "worth", "expensive", "cheap", "affordable"],
    "customer_support":["support", "service", "helpline", "refund", "return", "response", "agent"],
    "packaging":       ["packaging", "packed", "box", "unboxing"],
    "performance":     ["performance", "speed", "fast", "slow", "lag", "smooth", "working"],
}

CONTRAST_TOKENS = {"but", "however", "although", "though", "yet", "despite", "except"}


In [4]:
def load_reviews(path):
    try:
        df = pd.read_csv(path)
        assert "review_text" in df.columns and "sentiment_label" in df.columns
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {path}")
    except AssertionError:
        raise ValueError("Expected columns missing.")

df = load_reviews(REVIEWS_PATH)
vader = SentimentIntensityAnalyzer()
print("Dataset loaded:", df.shape)


Dataset loaded: (10199, 20)


## (a) Why is aspect-level harder than review-level?

In [5]:
explanation_a = """
Why Review-level (88% F1) vs Aspect-level (71% F1):

1. GRANULARITY:
   Review-level treats the whole text as one label. Majority sentiment wins.
   Aspect-level must assign a separate label per aspect within the same text.
   One review can be Positive (camera) + Negative (battery) simultaneously.

2. ASPECT DETECTION IS A SEPARATE SUBTASK:
   Before scoring sentiment, the system must first identify WHICH aspects are mentioned.
   This adds a second layer of error — if the aspect detector fails, the sentiment is wrong too.

3. SENTIMENT SCOPE IS LOCAL, NOT GLOBAL:
   "Amazing camera but atrocious battery" — the word 'but' flips sentiment scope.
   Review-level models see the full sentence and often predict Neutral or Mixed.
   Aspect-level must correctly scope "amazing" to "camera" and "atrocious" to "battery".

4. TRAINING DATA SCARCITY:
   Review-level labels are easy to collect (ratings are natural proxies).
   Aspect-level requires human annotators to tag each aspect + sentiment in each review.
   ShopSense has sentiment_label (review-level) but no aspect-level annotations natively.

5. IMPLICIT AND MULTI-WORD ASPECTS:
   "Drains fast" implies battery aspect without the word 'battery'.
   "Took 12 days" implies delivery aspect. These are hard to detect without context.
"""
print(explanation_a)



Why Review-level (88% F1) vs Aspect-level (71% F1):

1. GRANULARITY:
   Review-level treats the whole text as one label. Majority sentiment wins.
   Aspect-level must assign a separate label per aspect within the same text.
   One review can be Positive (camera) + Negative (battery) simultaneously.

2. ASPECT DETECTION IS A SEPARATE SUBTASK:
   Before scoring sentiment, the system must first identify WHICH aspects are mentioned.
   This adds a second layer of error — if the aspect detector fails, the sentiment is wrong too.

3. SENTIMENT SCOPE IS LOCAL, NOT GLOBAL:
   "Amazing camera but atrocious battery" — the word 'but' flips sentiment scope.
   Review-level models see the full sentence and often predict Neutral or Mixed.
   Aspect-level must correctly scope "amazing" to "camera" and "atrocious" to "battery".

4. TRAINING DATA SCARCITY:
   Review-level labels are easy to collect (ratings are natural proxies).
   Aspect-level requires human annotators to tag each aspect + sentiment 

## (b) How to improve from 71% to 80%+

In [6]:
improvement_plan = """
Strategies to move aspect-level F1 from 71% to 80%+:

1. SENTENCE SEGMENTATION BEFORE CLASSIFICATION:
   Split review into clauses at contrast words (but, however, although).
   Score each clause independently instead of the full review.
   Directly addresses the sentiment-scope problem.

2. EXPAND ASPECT KEYWORD DICTIONARIES:
   Add implicit aspect keywords: "drains" -> battery, "took 10 days" -> delivery.
   Use Word2Vec to find synonyms of each aspect term from the ShopSense corpus.

3. USE A PRE-TRAINED ABSA MODEL:
   Models fine-tuned for Aspect-Based Sentiment Analysis (like BERT-ABSA) significantly
   outperform rule-based systems. They can handle implicit aspects and negation scope.

4. HANDLE NEGATION SCOPE PROPERLY:
   Apply the NOT_ prepending trick before scoring each clause.
   "not unhelpful" = helpful; "not great" = bad.

5. ADD CONTRASTIVE CONJUNCTION FEATURES:
   Presence of "but", "however", "although" is a strong signal that the review
   contains mixed-sentiment aspects. Use this as a feature for the classifier.
"""
print(improvement_plan)



Strategies to move aspect-level F1 from 71% to 80%+:

1. SENTENCE SEGMENTATION BEFORE CLASSIFICATION:
   Split review into clauses at contrast words (but, however, although).
   Score each clause independently instead of the full review.
   Directly addresses the sentiment-scope problem.

2. EXPAND ASPECT KEYWORD DICTIONARIES:
   Add implicit aspect keywords: "drains" -> battery, "took 10 days" -> delivery.
   Use Word2Vec to find synonyms of each aspect term from the ShopSense corpus.

3. USE A PRE-TRAINED ABSA MODEL:
   Models fine-tuned for Aspect-Based Sentiment Analysis (like BERT-ABSA) significantly
   outperform rule-based systems. They can handle implicit aspects and negation scope.

4. HANDLE NEGATION SCOPE PROPERLY:
   Apply the NOT_ prepending trick before scoring each clause.
   "not unhelpful" = helpful; "not great" = bad.

5. ADD CONTRASTIVE CONJUNCTION FEATURES:
   Presence of "but", "however", "although" is a strong signal that the review
   contains mixed-sentiment as

## (c) Extract aspect-sentiment pairs from target review

In [7]:
def split_clauses(text, contrast_tokens=CONTRAST_TOKENS):
    pattern = r"\b(" + "|".join(contrast_tokens) + r")\b"
    parts = re.split(pattern, text, flags=re.IGNORECASE)
    clauses = [p.strip() for p in parts if p.strip() and p.lower() not in contrast_tokens]
    return clauses

def find_aspect_in_clause(clause, aspect_keywords=ASPECT_KEYWORDS):
    clause_lower = clause.lower()
    for aspect, keywords in aspect_keywords.items():
        if any(kw in clause_lower for kw in keywords):
            return aspect
    return "general"

def score_clause_sentiment(clause):
    score = vader.polarity_scores(clause)["compound"]
    if score >= 0.05:
        return "Positive", score
    elif score <= -0.05:
        return "Negative", score
    return "Neutral", score

def extract_aspect_sentiment_pairs(review_text):
    clauses = split_clauses(review_text)
    pairs = []
    for clause in clauses:
        aspect = find_aspect_in_clause(clause)
        sentiment, score = score_clause_sentiment(clause)
        pairs.append({
            "clause": clause,
            "aspect": aspect,
            "sentiment": sentiment,
            "vader_score": round(score, 4)
        })
    return pairs

pairs = extract_aspect_sentiment_pairs(TARGET_REVIEW)

print(f"Review: {TARGET_REVIEW}\n")
print("Aspect-Sentiment Pairs:")
for p in pairs:
    print(f"  Aspect: {p['aspect']:20s} | Sentiment: {p['sentiment']:10s} | Score: {p['vader_score']} | Clause: '{p['clause']}'")

sentiments = set(p["sentiment"] for p in pairs)
print(f"\nThis single review is simultaneously: {sentiments}")
print("Review-level classifier would collapse this to a single label — losing all nuance.")


Review: Amazing camera quality but the battery is atrocious and customer support was unhelpful.

Aspect-Sentiment Pairs:
  Aspect: camera               | Sentiment: Positive   | Score: 0.5859 | Clause: 'Amazing camera quality'
  Aspect: battery              | Sentiment: Positive   | Score: 0.4019 | Clause: 'the battery is atrocious and customer support was unhelpful.'

This single review is simultaneously: {'Positive'}
Review-level classifier would collapse this to a single label — losing all nuance.


## Run aspect extraction on real ShopSense reviews

In [8]:
sample = df["review_text"].dropna().sample(10, random_state=42).tolist()

print("=== Aspect-Sentiment on 10 Real Reviews ===")
for text in sample:
    pairs = extract_aspect_sentiment_pairs(str(text))
    if len(pairs) > 1:
        print(f"\nReview: {str(text)[:90]}")
        for p in pairs:
            print(f"  {p['aspect']:20s} -> {p['sentiment']} ({p['vader_score']})")


=== Aspect-Sentiment on 10 Real Reviews ===

Review: Product is okay. Nothing special but does the job. Average quality for the price.
  general              -> Negative (-0.092)
  quality              -> Neutral (0.0)

Review: Average experience. Delivery was on time but the product quality is just mediocre.
  delivery             -> Neutral (0.0)
  quality              -> Neutral (0.0)

Review: Outstanding quality. I was skeptical at first but this technical proved me wrong completel
  quality              -> Positive (0.4019)
  general              -> Negative (-0.4767)
